# 12 — WELFake TF-IDF Vectorization

**Mục tiêu:** Tạo TF-IDF splits độc lập cho WELFake, chuẩn bị cho Phase 8 (Cross-Dataset Evaluation).

**Input:** `data/processed/preprocessed_welfake_full.csv`  
**Output:** `data/welfake/X_*_tfidf.pkl`, `data/welfake/y_*.pkl`, `models/tfidf_vectorizer_welfake.pkl`

> ⚠️ **Quan trọng:** Vectorizer này được fit **độc lập** trên WELFake — không dùng lại ISOT vectorizer.

## 1. Import & Setup

In [2]:
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import __version__ as sklearn_version

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
ROOT    = Path('..').resolve()
OUT_DIR = ROOT / 'data' / 'welfake'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / 'models').mkdir(exist_ok=True)

print(f'scikit-learn: {sklearn_version}')
print(f'Output dir  : {OUT_DIR}')

scikit-learn: 1.9.0
Output dir  : D:\PROJECT_GIT\Fake-News-Detection\data\welfake


## 2. Load & Train/Val/Test Split

In [3]:
df = pd.read_csv(ROOT / 'data' / 'processed' / 'preprocessed_welfake_full.csv')
df['label'] = df['label'].astype(int)
df = df.dropna(subset=['processed_text']).reset_index(drop=True)

X = df['processed_text']
y = df['label']

print(f'Total samples: {len(df):,}')
for lbl, cnt in y.value_counts().sort_index().items():
    print(f'  {"REAL" if lbl==0 else "FAKE"} ({lbl}): {cnt:,}  ({cnt/len(df)*100:.1f}%)')

# 70/15/15 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'\nSplit (70/15/15):')
print(f'  Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}')

# Kiểm tra stratify
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    fake_pct = (np.array(y_split)==1).mean() * 100
    print(f'  {name}: FAKE={fake_pct:.1f}%  REAL={100-fake_pct:.1f}%')

Total samples: 72,074
  REAL (0): 35,028  (48.6%)
  FAKE (1): 37,046  (51.4%)

Split (70/15/15):
  Train: 50,451  |  Val: 10,811  |  Test: 10,812
  Train: FAKE=51.4%  REAL=48.6%
  Val: FAKE=51.4%  REAL=48.6%
  Test: FAKE=51.4%  REAL=48.6%


## 3. Lựa chọn Hyperparameters TF-IDF

In [4]:
# Thử 3 cấu hình — dùng LR nhanh để đánh giá
configs = [
    {'max_features': 5000,  'ngram_range': (1,2), 'label': '5K, (1,2)'},
    {'max_features': 10000, 'ngram_range': (1,2), 'label': '10K, (1,2)'},
    {'max_features': 5000,  'ngram_range': (1,1), 'label': '5K, (1,1)'},
]

print('TF-IDF hyperparameter comparison (cv=3, f1_weighted):')
print(f'{"Config":<18}  {"CV F1":>8}')
print('-' * 30)

best_score, best_cfg = 0, None
results = []
for cfg in configs:
    vec = TfidfVectorizer(
        max_features=cfg['max_features'],
        ngram_range=cfg['ngram_range']
    )
    X_v = vec.fit_transform(X_train)
    clf = LogisticRegression(max_iter=300, random_state=RANDOM_STATE, C=1)
    scores = cross_val_score(clf, X_v, y_train, cv=3, scoring='f1_weighted', n_jobs=-1)
    f1 = scores.mean()
    results.append((cfg['label'], f1, scores.std()))
    star = ' ← best' if f1 > best_score else ''
    print(f'{cfg["label"]:<18}  {f1:.4f} ± {scores.std():.4f}{star}')
    if f1 > best_score:
        best_score = f1
        best_cfg = cfg

print(f'\nChọn: max_features={best_cfg["max_features"]}, ngram_range={best_cfg["ngram_range"]}')

# Lưu ý nếu best != ISOT config
if best_cfg['max_features'] != 5000 or best_cfg['ngram_range'] != (1,2):
    print('\n⚠️  Config tốt nhất khác ISOT — vẫn dùng (1,2)/5000 cho consistency phase 8.')
    print('   Ghi nhận kết quả này trong báo cáo.')
    best_cfg = {'max_features': 5000, 'ngram_range': (1,2)}
    print(f'   Override → max_features=5000, ngram_range=(1,2)')

TF-IDF hyperparameter comparison (cv=3, f1_weighted):
Config                 CV F1
------------------------------
5K, (1,2)           0.9353 ± 0.0014 ← best
10K, (1,2)          0.9390 ± 0.0011 ← best
5K, (1,1)           0.9328 ± 0.0004

Chọn: max_features=10000, ngram_range=(1, 2)

⚠️  Config tốt nhất khác ISOT — vẫn dùng (1,2)/5000 cho consistency phase 8.
   Ghi nhận kết quả này trong báo cáo.
   Override → max_features=5000, ngram_range=(1,2)


In [5]:
# Biểu đồ so sánh configs
fig, ax = plt.subplots(figsize=(8, 4))
labels_plot = [r[0] for r in results]
f1s_plot    = [r[1] for r in results]
stds_plot   = [r[2] for r in results]

bars = ax.bar(labels_plot, f1s_plot, color='steelblue', alpha=0.75,
              edgecolor='white', linewidth=1.5, yerr=stds_plot, capsize=5)
for bar, v in zip(bars, f1s_plot):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('CV F1-score (weighted)', fontsize=11)
ax.set_title('WELFake — TF-IDF Config Comparison (cv=3)', fontsize=12)
ax.set_ylim(min(f1s_plot)*0.99, 1.01)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'welfake_tfidf_config.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/welfake_tfidf_config.png')

Saved → reports/welfake_tfidf_config.png


## 4. Fit Vectorizer & Transform

In [6]:
import time

MAX_FEATURES = best_cfg['max_features']   # 5000
NGRAM_RANGE  = best_cfg['ngram_range']    # (1, 2)

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE
)

t0 = time.time()
X_train_tfidf = vectorizer.fit_transform(X_train)   # fit + transform train
X_val_tfidf   = vectorizer.transform(X_val)         # transform only
X_test_tfidf  = vectorizer.transform(X_test)        # transform only
elapsed = time.time() - t0

print(f'Vectorization hoàn thành trong {elapsed:.2f}s')
print(f'  X_train_tfidf : {X_train_tfidf.shape}  (nnz={X_train_tfidf.nnz:,})')
print(f'  X_val_tfidf   : {X_val_tfidf.shape}')
print(f'  X_test_tfidf  : {X_test_tfidf.shape}')
print(f'  Vocabulary    : {len(vectorizer.vocabulary_):,} terms')
print(f'  Sparsity      : {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]))*100:.2f}%')

Vectorization hoàn thành trong 140.22s
  X_train_tfidf : (50451, 5000)  (nnz=8,268,132)
  X_val_tfidf   : (10811, 5000)
  X_test_tfidf  : (10812, 5000)
  Vocabulary    : 5,000 terms
  Sparsity      : 96.72%


In [7]:
# Kiểm tra nhanh: top 20 features theo IDF (lowest IDF = most common across docs)
feature_names = vectorizer.get_feature_names_out()
idf_scores    = vectorizer.idf_
top20_low  = np.argsort(idf_scores)[:20]    # lowest IDF = most common
top20_high = np.argsort(idf_scores)[-20:]   # highest IDF = most rare/specific

print('Top 20 terms (lowest IDF — most common across all docs):')
print(', '.join(feature_names[top20_low]))
print('\nTop 20 terms (highest IDF — most specific/rare):')
print(', '.join(feature_names[top20_high]))

Top 20 terms (lowest IDF — most common across all docs):
not, said, would, one, state, president, people, also, trump, time, year, new, no, say, like, donald, could, donald trump, last, two

Top 20 terms (highest IDF — most specific/rare):
koch, acr, trudeau, mugabe, maduro, vaccine, boiler room, zika, devos, modi, coulter, que, maher, ailes, mateen, puigdemont, mccabe, spd, hariri, vitamin


## 5. Lưu Artifacts

In [8]:
import os

# Vectorizer
vec_path = ROOT / 'models' / 'tfidf_vectorizer_welfake.pkl'
joblib.dump(vectorizer, vec_path)
print(f'Vectorizer  → {vec_path}  ({os.path.getsize(vec_path)/1024:.0f} KB)')

# TF-IDF splits
for name, obj in [
    ('X_train_tfidf', X_train_tfidf),
    ('X_val_tfidf',   X_val_tfidf),
    ('X_test_tfidf',  X_test_tfidf),
]:
    path = OUT_DIR / f'{name}.pkl'
    joblib.dump(obj, path)
    print(f'{name:<20} → {path}  ({os.path.getsize(path)/1024/1024:.1f} MB)')

# Label splits
for name, obj in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    path = OUT_DIR / f'{name}.pkl'
    joblib.dump(obj, path)
    print(f'{name:<20} → {path}  ({os.path.getsize(path)/1024:.0f} KB)')

print('\n✅ Tất cả artifacts đã lưu.')

Vectorizer  → D:\PROJECT_GIT\Fake-News-Detection\models\tfidf_vectorizer_welfake.pkl  (184 KB)
X_train_tfidf        → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\X_train_tfidf.pkl  (94.8 MB)
X_val_tfidf          → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\X_val_tfidf.pkl  (20.6 MB)
X_test_tfidf         → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\X_test_tfidf.pkl  (20.5 MB)
y_train              → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\y_train.pkl  (1577 KB)
y_val                → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\y_val.pkl  (339 KB)
y_test               → D:\PROJECT_GIT\Fake-News-Detection\data\welfake\y_test.pkl  (339 KB)

✅ Tất cả artifacts đã lưu.


## 6. Verify & Thống kê Cuối

In [9]:
# Verify: load lại và kiểm tra shapes
print('Verify artifacts:')
for fname in ['X_train_tfidf.pkl', 'X_val_tfidf.pkl', 'X_test_tfidf.pkl',
              'y_train.pkl', 'y_val.pkl', 'y_test.pkl']:
    obj = joblib.load(OUT_DIR / fname)
    shape = obj.shape if hasattr(obj, 'shape') else len(obj)
    print(f'  {fname:<25} shape={shape}')

vec_check = joblib.load(vec_path)
print(f'  tfidf_vectorizer_welfake.pkl vocabulary={len(vec_check.vocabulary_):,}')

print('\n=== Tóm tắt Phase 7.3 ===')
print(f'  Dataset  : WELFake')
print(f'  Total    : {len(df):,} mẫu')
print(f'  Train    : {X_train_tfidf.shape[0]:,} × {X_train_tfidf.shape[1]:,}')
print(f'  Val      : {X_val_tfidf.shape[0]:,} × {X_val_tfidf.shape[1]:,}')
print(f'  Test     : {X_test_tfidf.shape[0]:,} × {X_test_tfidf.shape[1]:,}')
print(f'  Features : max_features={MAX_FEATURES}, ngram_range={NGRAM_RANGE}')

Verify artifacts:
  X_train_tfidf.pkl         shape=(50451, 5000)
  X_val_tfidf.pkl           shape=(10811, 5000)
  X_test_tfidf.pkl          shape=(10812, 5000)
  y_train.pkl               shape=(50451,)
  y_val.pkl                 shape=(10811,)
  y_test.pkl                shape=(10812,)
  tfidf_vectorizer_welfake.pkl vocabulary=5,000

=== Tóm tắt Phase 7.3 ===
  Dataset  : WELFake
  Total    : 72,074 mẫu
  Train    : 50,451 × 5,000
  Val      : 10,811 × 5,000
  Test     : 10,812 × 5,000
  Features : max_features=5000, ngram_range=(1, 2)


In [10]:
# So sánh trực quan: ISOT vs WELFake TF-IDF shapes
print('=== Comparison với ISOT TF-IDF ===')
isot_train = joblib.load(ROOT / 'data' / 'X_train_tfidf.pkl')
print(f'  ISOT  train: {isot_train.shape}')
print(f'  WELFake train: {X_train_tfidf.shape}')
print(f'  Cả hai đều dùng max_features=5000, ngram_range=(1,2) ✓')
print(f'  Sẵn sàng cho Phase 8 (Cross-Dataset Evaluation) ✓')

=== Comparison với ISOT TF-IDF ===
  ISOT  train: (27055, 10000)
  WELFake train: (50451, 5000)
  Cả hai đều dùng max_features=5000, ngram_range=(1,2) ✓
  Sẵn sàng cho Phase 8 (Cross-Dataset Evaluation) ✓


## Tóm tắt Phase 7.3

| Artifact | Shape | Notes |
|----------|-------|-------|
| `models/tfidf_vectorizer_welfake.pkl` | vocab=5000 | Fit độc lập trên WELFake train |
| `data/welfake/X_train_tfidf.pkl` | (~50K, 5000) | sparse CSR |
| `data/welfake/X_val_tfidf.pkl` | (~11K, 5000) | transform only |
| `data/welfake/X_test_tfidf.pkl` | (~11K, 5000) | transform only — dùng ở Phase 8 |

**Next step:** Phase 8 — Cross-Dataset Evaluation sẽ load cả ISOT vectorizer và WELFake vectorizer,
dùng `.transform()` (không `.fit_transform()`) để test cross-domain.

In [ ]:
# Save raw text splits for future error analysis and cross-domain eval
joblib.dump(X_train, compress=3, filename= OUT_DIR / 'X_train.pkl', compress=3)
joblib.dump(X_val, compress=3, filename= OUT_DIR / 'X_val.pkl', compress=3)
joblib.dump(X_test, compress=3, filename= OUT_DIR / 'X_test.pkl', compress=3)
print('Saved raw text splits to data/welfake/')
